In [9]:
import pandas  as pd
df=pd.read_csv(r"D:\3\city_master.csv")
df.head()

,city_id,city,country,region,rent_1br_center_usd,rent_1br_outside_usd,meal_inexpensive_usd,monthly_transport_pass_usd,groceries_monthly_usd,coworking_monthly_usd,...,english_proficiency,healthcare_index,internet_estimated,remote_work_visa,min_monthly_income_usd,max_stay_months,allows_renewal,rent_growth_12m_pct,monthly_cost_usd,visa_status
0,1,Lisbon,Portugal,Europe,1398.0,1052.0,9.3,34.8,295.0,172.0,...,85.0,79.0,False,Yes,3000.0,6.0,Yes,7.6,1972.0,Visa available
1,2,Barcelona,Spain,Europe,1329.0,959.0,9.6,49.2,271.0,178.0,...,57.0,85.0,False,Yes,2500.0,24.0,Yes,7.7,1882.0,Visa available
2,3,Madrid,Spain,Europe,1133.0,858.0,9.1,31.1,248.0,203.0,...,60.0,72.0,False,Yes,2500.0,24.0,Yes,4.3,1671.0,Visa available
3,4,Berlin,Germany,Europe,1309.0,923.0,9.8,41.5,279.0,189.0,...,76.0,87.0,False,Yes,2500.0,12.0,No,8.9,1871.0,Visa available
4,5,Prague,Czechia,Europe,997.0,756.0,8.2,35.0,218.0,112.0,...,68.0,71.0,False,Yes,4500.0,12.0,Yes,3.2,1412.0,Visa available


In [3]:
import pandas as pd
m = pd.read_csv(r"D:\3\city_master.csv")
print(m.isna().sum()[lambda s: s > 0])

min_monthly_income_usd    28
max_stay_months           28
dtype: int64


In [6]:
import pandas as pd
m["visa_status"] = m.remote_work_visa.map({"Yes": "Visa available", "No": "No remote-work visa"})
m.to_csv(r"D:\3\city_master.csv", index=False)

In [7]:
# 1. the table should have exactly 63 cities, one row each
print("rows:", len(m), "| unique cities:", m.city.is_unique)

# 2. blanks should appear ONLY where there is no visa
print(pd.crosstab(m.min_monthly_income_usd.isna(), m.remote_work_visa,
                  rownames=["income is blank"]))

# 3. the same check, written as a pass/fail test
assert m[m.remote_work_visa == "Yes"][["min_monthly_income_usd", "max_stay_months"]].notna().all().all(), "a visa country has a blank"
assert m[m.remote_work_visa == "No"][["min_monthly_income_usd", "max_stay_months"]].isna().all().all(), "a no-visa country has a value"
print("All checks passed")

rows: 63 | unique cities: True
remote_work_visa  No  Yes
income is blank          
False              0   35
True              28    0
All checks passed


In [12]:
import pandas as pd
from sklearn.cluster import KMeans

# 1. scale every metric to 0-100 (higher = better)
def scale(s, invert=False):
    z = (s - s.min()) / (s.max() - s.min()) * 100
    return 100 - z if invert else z

df["cost_score"]     = scale(df.monthly_cost_usd, invert=True)   # cheaper = better
df["internet_score"] = scale(df.internet_mbps)
df["safety_score"]   = scale(df.safety_index)
df["english_score"]  = scale(df.english_proficiency)

# 2. weighted score (weights must add up to 1)
def add_score(data, weights):
    assert abs(sum(weights.values()) - 1) < 1e-9, "weights must add up to 1"
    out = data.copy()
    out["score"] = sum(out[f"{k}_score"] * w for k, w in weights.items())
    out["rank"] = out.score.rank(ascending=False, method="min").astype(int)
    return out.sort_values("rank")

base = {"cost": 0.40, "internet": 0.20, "safety": 0.25, "english": 0.15}
ranked = add_score(df, base)
print("TOP 10 (cost 40%)\n", ranked[["rank", "city", "country", "score"]].head(10).round(1).to_string(index=False))

# 3. what changes if cost matters less?
low_cost = add_score(df, {"cost": 0.10, "internet": 0.30, "safety": 0.35, "english": 0.25})
print("\nTOP 10 (cost 10%)\n", low_cost[["rank", "city", "country", "score"]].head(10).round(1).to_string(index=False))
moved_out = set(ranked.head(10).city) - set(low_cost.head(10).city)
print("\nLeft the top 10:", sorted(moved_out))

# 4. cluster cities into 4 types
features = ["cost_score", "internet_score", "safety_score", "english_score"]
df["cluster"] = KMeans(n_clusters=4, n_init=10, random_state=42).fit_predict(df[features])

# name each cluster: rank by quality (best = 1) + a cost label from its real average monthly cost
prof = df.groupby("cluster")[features].mean()
prof["quality"] = prof[["internet_score", "safety_score", "english_score"]].mean(axis=1)
prof["avg_cost"] = df.groupby("cluster").monthly_cost_usd.mean()
quality_rank = prof.quality.rank(ascending=False).astype(int)
tiers = {1: "Top quality", 2: "Good quality", 3: "Basic quality", 4: "Lowest quality"}
cost_label = lambda c: "low cost" if c < 1000 else ("mid cost" if c < 1500 else "high cost")
names = {c: f"{quality_rank[c]}. {tiers[quality_rank[c]]}, {cost_label(prof.avg_cost[c])}" for c in prof.index}
df["city_type"] = df.cluster.map(names)
print("\nCITY TYPES")
print(df.groupby("city_type").agg(cities=("city", "count"), avg_cost=("monthly_cost_usd", "mean")).round(0).sort_index())
for t, g in df.groupby("city_type"):
    print(f"  {t}: {', '.join(g.city.head(6))}")

# 5. save for the dashboard
final = ranked.merge(df[["city_id", "cluster", "city_type"]], on="city_id")
final.to_csv(r"D:\city_scores.csv", index=False)
print("\nsaved clean/city_scores.csv", final.shape)

TOP 10 (cost 40%)
  rank       city              country  score
    1    Tallinn              Estonia   70.6
    2      Seoul          South Korea   67.3
    3 Chiang Mai             Thailand   67.1
    4      Dubai United Arab Emirates   67.1
    5     Taipei               Taiwan   66.8
    6       Doha                Qatar   64.1
    7   Auckland          New Zealand   63.6
    8     Lisbon             Portugal   63.4
    9   Budapest              Hungary   62.1
   10  Singapore            Singapore   62.0

TOP 10 (cost 10%)
  rank      city              country  score
    1 Singapore            Singapore   84.6
    2     Dubai United Arab Emirates   75.6
    3    Zurich          Switzerland   74.9
    4 Hong Kong            Hong Kong   74.0
    5    Austin        United States   67.7
    6  Auckland          New Zealand   66.7
    7   Tallinn              Estonia   65.6
    8      Doha                Qatar   64.7
    9     Seoul          South Korea   63.4
   10    Lisbon           